# Uzbek NER — thresholds, ансамбль и исправление границ

Ноутбук сравнивает два уже обученных NER-checkpoint на размеченной dev-выборке:

1. сохраняет конфигурации и entity-level предсказания с вероятностями;
2. независимо подбирает threshold каждого класса `ORG`, `NAME`, `GEO` по class F1;
3. применяет thresholds и строит precision-oriented ансамбль exact-span голосованием;
4. отдельно повторяет оценку после консервативного исправления границ, попавших внутрь слова.

Thresholds подбираются только на dev и затем должны быть заморожены для test. Вторая часть использует те же thresholds и не подгоняет их повторно после исправления границ.


In [1]:
# Если в образе Kaggle не хватает зависимостей, раскомментируйте и перезапустите kernel.
%pip install -q "transformers>=4.48" "tokenizers>=0.20" "accelerate>=1.0" "tqdm>=4.66" pandas



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Конфигурация

Укажите две директории с полностью обученными NER-моделями и конкретный JSONL с dev-разметкой. Для нормализованной оценки укажите `preprocessing/dev.jsonl` или соответствующий `dev.jsonl` Kaggle Dataset.


In [ ]:
from pathlib import Path
import csv
import gc
import json
import math
import re
import sys
from collections import Counter

import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForTokenClassification


INPUT_ROOT = Path("/Users/tsagoll/AI Talent Hub/Hack")
OUTPUT_ROOT = Path("/Users/tsagoll/AI Talent Hub/Hack/ner_ensemble_runs")

MODEL_SPECS = [
    {
        "name": "normalized",
        "path": Path("/Users/tsagoll/AI Talent Hub/Hack/normalized_mmBERT-base/model"),
        "max_length": None,
        "stride": None,
    },
    {
        "name": "big_augmented",
        "path": Path("/Users/tsagoll/AI Talent Hub/Hack/augmented_mmBERT-base/model"),
        "max_length": None,
        "stride": None,
    },
]

DEV_PATH = Path(
    "/Users/tsagoll/AI Talent Hub/Hack/data/preprocessing/dev.jsonl"
)
PUBLIC_TEST_PATH = INPUT_ROOT / "public_test_inputs.jsonl"

EXPERIMENT_NAME = "mmbert_pair_dev"
REUSE_EXISTING_DEV_PREDICTIONS = True
NORMALIZE_PUBLIC_APOSTROPHES = True
PUBLIC_SUBMISSION_MODE = "labelwise"  # intersection, union или labelwise
LABELWISE_MODEL_BY_CLASS = {
    "ORG": "normalized",
    "NAME": "big_augmented",
    "GEO": "normalized",
}

EVAL_BATCH_SIZE = 4
DEVICE = "mps"  # "cuda", "mps" или "cpu"
CONFIDENCE_FIELD = "probability"
THRESHOLD_OBJECTIVE = "f1"
MIN_VOTES = 2  # Для двух моделей это exact-span intersection.

DEFAULT_MAX_LENGTH = 512
DEFAULT_STRIDE = 128
LABELS = ("ORG", "NAME", "GEO")

EXPERIMENT_DIR = OUTPUT_ROOT / EXPERIMENT_NAME
CONFIG_DIR = EXPERIMENT_DIR / "configs"
RAW_DIR = EXPERIMENT_DIR / "raw"
THRESHOLD_DIR = EXPERIMENT_DIR / "thresholds"
ENSEMBLE_DIR = EXPERIMENT_DIR / "ensemble"
BOUNDARY_DIR = EXPERIMENT_DIR / "word_boundary_fix"
PUBLIC_TEST_DIR = EXPERIMENT_DIR / "public_test"

for directory in (
    CONFIG_DIR,
    RAW_DIR,
    THRESHOLD_DIR,
    ENSEMBLE_DIR,
    BOUNDARY_DIR,
    PUBLIC_TEST_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

if len(MODEL_SPECS) != 2:
    raise ValueError("MODEL_SPECS должен содержать ровно две модели")
model_names = [spec["name"] for spec in MODEL_SPECS]
if len(set(model_names)) != 2:
    raise ValueError("Имена моделей должны различаться")
if any(not re.fullmatch(r"[A-Za-z0-9_-]+", name) for name in model_names):
    raise ValueError("Имена моделей могут содержать только буквы, цифры, _ и -")
if not 1 <= MIN_VOTES <= len(MODEL_SPECS):
    raise ValueError("MIN_VOTES вне допустимого диапазона")

for spec in MODEL_SPECS:
    if not Path(spec["path"]).is_dir():
        raise FileNotFoundError(f"Не найдена директория модели: {spec['path']}")
if not DEV_PATH.is_file():
    raise FileNotFoundError(f"Не найдена dev-разметка: {DEV_PATH}")
if not PUBLIC_TEST_PATH.is_file():
    raise FileNotFoundError(f"Не найден public test: {PUBLIC_TEST_PATH}")
if PUBLIC_SUBMISSION_MODE not in {"intersection", "union", "labelwise"}:
    raise ValueError("Неизвестный PUBLIC_SUBMISSION_MODE")
if set(LABELWISE_MODEL_BY_CLASS) != set(LABELS):
    raise ValueError("LABELWISE_MODEL_BY_CLASS должен задавать ORG, NAME и GEO")
if not set(LABELWISE_MODEL_BY_CLASS.values()) <= set(model_names):
    raise ValueError("LABELWISE_MODEL_BY_CLASS ссылается на неизвестную модель")

print("EXPERIMENT_DIR:", EXPERIMENT_DIR)
print("DEV_PATH:       ", DEV_PATH)
print("PUBLIC_TEST_PATH:", PUBLIC_TEST_PATH)
for spec in MODEL_SPECS:
    print(f"{spec['name']:<16}", spec["path"])


## 2. Baseline-функции, данные и конфиги

Используется актуальная оконная токенизация baseline. Локальный адаптер, как и в тренировочном ноутбуке, убирает внешние пробелы из offsets mmBERT.


In [3]:
COMMON_PATH = next(INPUT_ROOT.rglob("baseline/common.py"))
PROJECT_DIR = COMMON_PATH.parent.parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import baseline.common as common_module
import baseline.predict as predict_module
from baseline.common import TAGS, TAG_TO_ID, load_fast_tokenizer, read_records, validate_window
from baseline.predict import _build_windows, _model_labels, _predict_token_scores
from scripts.evaluate import calculate_metrics, print_metrics


def trim_whitespace_offsets(text, offsets):
    trimmed = []
    for start, end in offsets:
        start, end = int(start), int(end)
        while start < end and text[start].isspace():
            start += 1
        while end > start and text[end - 1].isspace():
            end -= 1
        trimmed.append((start, end))
    return trimmed


if not hasattr(common_module, "_untrimmed_tokenize_windows"):
    common_module._untrimmed_tokenize_windows = common_module.tokenize_windows


def tokenize_windows_with_trimmed_offsets(tokenizer, text, *, max_length, stride):
    windows = common_module._untrimmed_tokenize_windows(
        tokenizer,
        text,
        max_length=max_length,
        stride=stride,
    )
    return [
        (feature, trim_whitespace_offsets(text, offsets))
        for feature, offsets in windows
    ]


common_module.tokenize_windows = tokenize_windows_with_trimmed_offsets
predict_module.tokenize_windows = tokenize_windows_with_trimmed_offsets

device = torch.device(DEVICE)

dev_records = read_records(DEV_PATH, require_entities=True)
records_by_hash = {record["hash"]: record for record in dev_records}
DEV_GOLD = {
    record["hash"]: {
        "entities": {
            (entity["label"], entity["start"], entity["end"])
            for entity in record["entities"]
        }
    }
    for record in dev_records
}

ID_TO_TAG = {index: tag for index, tag in enumerate(TAGS)}
print(f"Dev records: {len(dev_records):,}")


Dev records: 1,500


In [ ]:
def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )


def write_jsonl(path, records):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as stream:
        for record in records:
            stream.write(json.dumps(record, ensure_ascii=False, separators=(",", ":")))
            stream.write("\n")


def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def competition_predictions(records):
    return [
        {
            "hash": record["hash"],
            "entities": [
                {
                    "label": entity["label"],
                    "start": entity["start"],
                    "end": entity["end"],
                }
                for entity in record["entities"]
            ],
        }
        for record in records
    ]


def predictions_for_metrics(records):
    return {
        record["hash"]: {
            (entity["label"], entity["start"], entity["end"])
            for entity in record["entities"]
        }
        for record in records
    }


def evaluate_records(records):
    return calculate_metrics(DEV_GOLD, predictions_for_metrics(records))


experiment_config = {
    "experiment_name": EXPERIMENT_NAME,
    "dev_path": str(DEV_PATH),
    "models": [
        {
            **{key: value for key, value in spec.items() if key != "path"},
            "path": str(spec["path"]),
        }
        for spec in MODEL_SPECS
    ],
    "eval_batch_size": EVAL_BATCH_SIZE,
    "device": DEVICE,
    "confidence_field": CONFIDENCE_FIELD,
    "reuse_existing_dev_predictions": REUSE_EXISTING_DEV_PREDICTIONS,
    "threshold_objective": THRESHOLD_OBJECTIVE,
    "min_votes": MIN_VOTES,
    "boundary_fix": "expand only boundaries inside a continuous alphanumeric sequence",
    "public_test_path": str(PUBLIC_TEST_PATH),
    "normalize_public_apostrophes": NORMALIZE_PUBLIC_APOSTROPHES,
    "public_submission_mode": PUBLIC_SUBMISSION_MODE,
    "labelwise_model_by_class": LABELWISE_MODEL_BY_CLASS,
}
write_json(CONFIG_DIR / "experiment_config.json", experiment_config)

for spec in MODEL_SPECS:
    model_dir = Path(spec["path"])
    payload = {
        "name": spec["name"],
        "source_path": str(model_dir),
        "transformers_config": read_json(model_dir / "config.json", {}),
        "baseline_config": read_json(model_dir / "baseline_config.json", {}),
    }
    write_json(CONFIG_DIR / f"{spec['name']}.json", payload)

print("Конфиги сохранены в", CONFIG_DIR)


## 3. Инференс двух моделей с entity-level вероятностями

Для каждого декодированного span сохраняются:

- `probability`: минимальная среди токенов вероятность выбранного класса `P(B-class)+P(I-class)`;
- `mean_probability`: средняя вероятность выбранного класса;
- `min_tag_probability` и `mean_tag_probability`: уверенность именно в выбранных BIO-тегах;
- `class_probabilities`: средние вероятности трёх entity-классов.

Threshold оптимизируется по консервативному полю `probability`.


In [ ]:
def decode_records_with_probabilities(records, scores, id2label):
    predictions = []

    for record, record_scores in zip(records, scores, strict=True):
        tokens = []
        for (start, end), (score_sum, count) in sorted(record_scores.items()):
            probabilities = (score_sum / count).float()
            label_id = int(probabilities.argmax().item())
            tag = id2label[label_id]
            class_probabilities = {
                label: float(
                    probabilities[TAG_TO_ID[f"B-{label}"]]
                    + probabilities[TAG_TO_ID[f"I-{label}"]]
                )
                for label in LABELS
            }
            tokens.append(
                {
                    "start": start,
                    "end": end,
                    "tag": tag,
                    "tag_probability": float(probabilities[label_id]),
                    "class_probabilities": class_probabilities,
                }
            )

        entities = []
        current = None

        def flush():
            nonlocal current
            if current is None:
                return
            chosen = current.pop("chosen_class_probabilities")
            tag_scores = current.pop("tag_probabilities")
            class_vectors = current.pop("class_vectors")
            current["probability"] = min(chosen)
            current["mean_probability"] = sum(chosen) / len(chosen)
            current["min_tag_probability"] = min(tag_scores)
            current["mean_tag_probability"] = sum(tag_scores) / len(tag_scores)
            current["class_probabilities"] = {
                label: sum(vector[label] for vector in class_vectors) / len(class_vectors)
                for label in LABELS
            }
            entities.append(current)
            current = None

        for token in tokens:
            tag = token["tag"]
            if tag == "O":
                flush()
                continue

            prefix, separator, label = tag.partition("-")
            if separator != "-" or prefix not in {"B", "I"} or label not in LABELS:
                raise ValueError(f"Неподдерживаемый BIO-тег: {tag!r}")

            if prefix == "B" or current is None or current["label"] != label:
                flush()
                current = {
                    "label": label,
                    "start": token["start"],
                    "end": token["end"],
                    "chosen_class_probabilities": [],
                    "tag_probabilities": [],
                    "class_vectors": [],
                }
            else:
                current["end"] = max(current["end"], token["end"])

            current["chosen_class_probabilities"].append(
                token["class_probabilities"][label]
            )
            current["tag_probabilities"].append(token["tag_probability"])
            current["class_vectors"].append(token["class_probabilities"])

        flush()
        predictions.append({"hash": record["hash"], "entities": entities})

    return predictions


def resolve_window_parameters(spec):
    baseline_config = read_json(Path(spec["path"]) / "baseline_config.json", {})
    max_length = spec.get("max_length")
    stride = spec.get("stride")
    if max_length is None:
        max_length = int(baseline_config.get("max_length", DEFAULT_MAX_LENGTH))
    if stride is None:
        stride = int(baseline_config.get("stride", DEFAULT_STRIDE))
    return int(max_length), int(stride)


def predict_records_with_model(spec, records, split_name):
    name = spec["name"]
    model_dir = Path(spec["path"])
    max_length, stride = resolve_window_parameters(spec)

    print(f"\n=== {name}: {split_name} ===")
    print("Model:", model_dir)
    print("Window:", max_length, "stride:", stride)

    tokenizer = load_fast_tokenizer(str(model_dir))
    validate_window(tokenizer, max_length, stride)
    windows = _build_windows(
        records,
        tokenizer,
        max_length=max_length,
        stride=stride,
    )
    model = AutoModelForTokenClassification.from_pretrained(model_dir).to(device)
    id2label = _model_labels(model)
    scores = _predict_token_scores(
        model,
        tokenizer,
        windows,
        len(records),
        batch_size=EVAL_BATCH_SIZE,
        device=device,
    )
    predictions = decode_records_with_probabilities(records, scores, id2label)
    inference_config = {
        "split": split_name,
        "model_path": str(model_dir),
        "max_length": max_length,
        "stride": stride,
        "batch_size": EVAL_BATCH_SIZE,
        "records": len(records),
        "windows": len(windows),
    }

    del scores, model, tokenizer, windows
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.empty_cache()
    return predictions, inference_config


def predict_one_model(spec):
    name = spec["name"]
    predictions, inference_config = predict_records_with_model(
        spec,
        dev_records,
        "dev",
    )
    metrics = evaluate_records(predictions)
    write_jsonl(RAW_DIR / f"{name}_predictions_with_scores.jsonl", predictions)
    write_json(RAW_DIR / f"{name}_metrics_no_threshold.json", metrics)
    write_json(RAW_DIR / f"{name}_inference_config.json", inference_config)
    print_metrics(metrics)
    return predictions


In [ ]:
raw_paths = {
    spec["name"]: RAW_DIR / f"{spec['name']}_predictions_with_scores.jsonl"
    for spec in MODEL_SPECS
}

if REUSE_EXISTING_DEV_PREDICTIONS and all(path.is_file() for path in raw_paths.values()):
    raw_predictions = {
        name: read_jsonl(path)
        for name, path in raw_paths.items()
    }
    print("Reused existing dev predictions from", RAW_DIR)
else:
    raw_predictions = {
        spec["name"]: predict_one_model(spec)
        for spec in MODEL_SPECS
    }
    print("\nRaw predictions saved in", RAW_DIR)


## 4. Подбор отдельных thresholds для ORG, NAME и GEO

Для каждого класса predictions сортируются по вероятности. Таблица содержит метрики при каждом реально достижимом threshold. Лучший threshold максимизирует class F1; при равном F1 выбирается вариант с большим precision, затем более высокий threshold.


In [7]:
def metric_values(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


def build_threshold_curve(records, label):
    gold_count = sum(
        entity[0] == label
        for record in DEV_GOLD.values()
        for entity in record["entities"]
    )
    candidates = []
    for record in records:
        gold_entities = DEV_GOLD[record["hash"]]["entities"]
        for entity in record["entities"]:
            if entity["label"] != label:
                continue
            key = (entity["label"], entity["start"], entity["end"])
            candidates.append(
                (float(entity[CONFIDENCE_FIELD]), key in gold_entities)
            )

    candidates.sort(key=lambda item: item[0], reverse=True)
    rows = []
    tp = fp = 0
    rows.append(
        {
            "label": label,
            "threshold": 1.000001,
            **metric_values(0, 0, gold_count),
            "predicted": 0,
            "gold": gold_count,
        }
    )

    index = 0
    while index < len(candidates):
        threshold = candidates[index][0]
        while index < len(candidates) and candidates[index][0] == threshold:
            if candidates[index][1]:
                tp += 1
            else:
                fp += 1
            index += 1
        values = metric_values(tp, fp, gold_count - tp)
        rows.append(
            {
                "label": label,
                "threshold": threshold,
                **values,
                "predicted": tp + fp,
                "gold": gold_count,
            }
        )

    if not rows or rows[-1]["threshold"] != 0.0:
        values = metric_values(tp, fp, gold_count - tp)
        rows.append(
            {
                "label": label,
                "threshold": 0.0,
                **values,
                "predicted": tp + fp,
                "gold": gold_count,
            }
        )
    return rows


def choose_best_threshold(rows):
    return max(
        rows,
        key=lambda row: (
            row[THRESHOLD_OBJECTIVE],
            row["precision"],
            row["threshold"],
        ),
    )


def write_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = list(rows[0])
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


best_thresholds = {}
best_rows = []

for spec in MODEL_SPECS:
    name = spec["name"]
    curves = []
    best_thresholds[name] = {}
    model_best = {}

    for label in LABELS:
        curve = build_threshold_curve(raw_predictions[name], label)
        curves.extend(curve)
        best = choose_best_threshold(curve)
        best_thresholds[name][label] = best["threshold"]
        model_best[label] = best
        best_rows.append({"model": name, **best})

    write_csv(THRESHOLD_DIR / f"{name}_threshold_metrics.csv", curves)
    write_json(THRESHOLD_DIR / f"{name}_threshold_metrics.json", curves)
    write_json(
        THRESHOLD_DIR / f"{name}_best_thresholds.json",
        {
            "confidence_field": CONFIDENCE_FIELD,
            "objective": THRESHOLD_OBJECTIVE,
            "thresholds": best_thresholds[name],
            "best_metrics": model_best,
        },
    )

best_threshold_table = pd.DataFrame(best_rows)[
    ["model", "label", "threshold", "precision", "recall", "f1", "tp", "fp", "fn"]
]
display(best_threshold_table)
best_threshold_table.to_csv(THRESHOLD_DIR / "best_thresholds_summary.csv", index=False)
print("Threshold reports saved in", THRESHOLD_DIR)


,model,label,threshold,precision,recall,f1,tp,fp,fn
0,normalized,ORG,0.790298,0.891384,0.820985,0.854738,2183,266,476
1,normalized,NAME,0.758168,0.912578,0.882277,0.897172,2046,196,273
2,normalized,GEO,0.706420,0.930953,0.872426,0.900740,2373,176,347
3,big_augmented,ORG,0.654314,0.874162,0.833396,0.853292,2216,319,443
4,big_augmented,NAME,0.540957,0.915970,0.897801,0.906794,2082,191,237
5,big_augmented,GEO,0.687555,0.917270,0.884559,0.900618,2406,217,314


Threshold reports saved in /Users/tsagoll/AI Talent Hub/Hack/ner_ensemble_runs/mmbert_pair_dev/thresholds


In [11]:
best_threshold_table.groupby('model')['f1'].mean()

model
big_augmented    0.886901
normalized       0.884217
Name: f1, dtype: float64

## 5. Thresholded-модели и ансамбль

Сначала к каждой модели применяются её собственные class thresholds. Затем span получает голос модели только при полном совпадении `label + start + end`. При двух моделях и `MIN_VOTES = 2` ансамбль является их exact-span пересечением.


In [ ]:
def apply_thresholds(records, thresholds):
    return [
        {
            "hash": record["hash"],
            "entities": [
                dict(entity)
                for entity in record["entities"]
                if float(entity[CONFIDENCE_FIELD]) >= thresholds[entity["label"]]
            ],
        }
        for record in records
    ]


def vote_predictions(predictions_by_model, min_votes, source_records=None):
    if source_records is None:
        source_records = dev_records
    indexed = {
        model_name: {record["hash"]: record["entities"] for record in records}
        for model_name, records in predictions_by_model.items()
    }
    output = []

    for record in source_records:
        record_hash = record["hash"]
        votes = {}
        for model_name, records in indexed.items():
            for entity in records[record_hash]:
                key = (entity["label"], entity["start"], entity["end"])
                votes.setdefault(key, {})[model_name] = entity

        entities = []
        for (label, start, end), model_entities in votes.items():
            if len(model_entities) < min_votes:
                continue
            probabilities = {
                model_name: float(entity[CONFIDENCE_FIELD])
                for model_name, entity in model_entities.items()
            }
            entities.append(
                {
                    "label": label,
                    "start": start,
                    "end": end,
                    "probability": sum(probabilities.values()) / len(probabilities),
                    "votes": len(model_entities),
                    "model_probabilities": probabilities,
                }
            )
        entities.sort(key=lambda entity: (entity["start"], entity["end"], entity["label"]))
        output.append({"hash": record_hash, "entities": entities})
    return output


thresholded_predictions = {}
thresholded_summary = []

for spec in MODEL_SPECS:
    name = spec["name"]
    predictions = apply_thresholds(raw_predictions[name], best_thresholds[name])
    metrics = evaluate_records(predictions)
    thresholded_predictions[name] = predictions
    write_jsonl(THRESHOLD_DIR / f"{name}_thresholded_predictions_with_scores.jsonl", predictions)
    write_jsonl(
        THRESHOLD_DIR / f"{name}_thresholded_predictions.jsonl",
        competition_predictions(predictions),
    )
    write_json(THRESHOLD_DIR / f"{name}_thresholded_metrics.json", metrics)
    thresholded_summary.append(
        {
            "model": name,
            **{f"micro_{key}": metrics["micro"][key] for key in ("precision", "recall", "f1")},
        }
    )
    print(f"\n{name} after thresholds")
    print_metrics(metrics)

ensemble_predictions = vote_predictions(thresholded_predictions, MIN_VOTES)
ensemble_metrics = evaluate_records(ensemble_predictions)
write_jsonl(ENSEMBLE_DIR / "ensemble_predictions_with_scores.jsonl", ensemble_predictions)
write_jsonl(
    ENSEMBLE_DIR / "ensemble_predictions.jsonl",
    competition_predictions(ensemble_predictions),
)
write_json(ENSEMBLE_DIR / "ensemble_metrics.json", ensemble_metrics)

print("\nThresholded ensemble")
print_metrics(ensemble_metrics)
display(pd.DataFrame(thresholded_summary))
print("Ensemble outputs saved in", ENSEMBLE_DIR)


## 6. Исправление границ внутри слова

Исправление выполняется после применения уже найденных thresholds и до голосования. Граница расширяется только тогда, когда стоит между двумя буквенно-цифровыми символами. Это исправляет subword-span вроде `oshkent` внутри `Toshkent`, но не поглощает соседние пробелы, кавычки, дефисы или отдельные слова.

После расширения возможные дубли и пересечения разрешаются в пользу entity с большей вероятностью. Результаты сохраняются в отдельной поддиректории `word_boundary_fix`.


In [ ]:
def expand_boundaries_inside_alnum(text, entity):
    original_start = int(entity["start"])
    original_end = int(entity["end"])
    start, end = original_start, original_end

    if (
        0 < start < len(text)
        and text[start - 1].isalnum()
        and text[start].isalnum()
    ):
        while start > 0 and text[start - 1].isalnum():
            start -= 1

    if (
        0 < end < len(text)
        and text[end - 1].isalnum()
        and text[end].isalnum()
    ):
        while end < len(text) and text[end].isalnum():
            end += 1

    fixed = dict(entity)
    fixed["start"] = start
    fixed["end"] = end
    if (start, end) != (original_start, original_end):
        fixed["boundary_fixed"] = True
        fixed["original_start"] = original_start
        fixed["original_end"] = original_end
    return fixed


def spans_overlap(left, right):
    return left["start"] < right["end"] and right["start"] < left["end"]


def resolve_expansion_conflicts(entities):
    unique = {}
    for entity in entities:
        key = (entity["label"], entity["start"], entity["end"])
        previous = unique.get(key)
        if previous is None or entity[CONFIDENCE_FIELD] > previous[CONFIDENCE_FIELD]:
            unique[key] = entity

    ranked = sorted(
        unique.values(),
        key=lambda entity: (
            -float(entity[CONFIDENCE_FIELD]),
            -(entity["end"] - entity["start"]),
            entity["start"],
            entity["label"],
        ),
    )
    kept = []
    dropped = 0
    for entity in ranked:
        if any(spans_overlap(entity, accepted) for accepted in kept):
            dropped += 1
        else:
            kept.append(entity)
    kept.sort(key=lambda entity: (entity["start"], entity["end"], entity["label"]))
    return kept, dropped


def apply_word_boundary_fix(records, source_records_by_hash=None):
    if source_records_by_hash is None:
        source_records_by_hash = records_by_hash
    output = []
    changed = dropped = 0
    for record in records:
        text = source_records_by_hash[record["hash"]]["text"]
        expanded = []
        for entity in record["entities"]:
            fixed = expand_boundaries_inside_alnum(text, entity)
            changed += int(fixed.get("boundary_fixed", False))
            expanded.append(fixed)
        resolved, record_dropped = resolve_expansion_conflicts(expanded)
        dropped += record_dropped
        output.append({"hash": record["hash"], "entities": resolved})
    return output, {"changed_entities": changed, "dropped_after_overlap": dropped}


In [14]:
fixed_predictions = {}
boundary_report = {"models": {}}
fixed_summary = []

for spec in MODEL_SPECS:
    name = spec["name"]
    predictions, report = apply_word_boundary_fix(thresholded_predictions[name])
    metrics = evaluate_records(predictions)
    fixed_predictions[name] = predictions
    boundary_report["models"][name] = report

    model_dir = BOUNDARY_DIR / name
    write_jsonl(model_dir / "predictions_with_scores.jsonl", predictions)
    write_jsonl(model_dir / "predictions.jsonl", competition_predictions(predictions))
    write_json(model_dir / "metrics.json", metrics)
    write_json(model_dir / "boundary_fix_report.json", report)
    fixed_summary.append(
        {
            "model": name,
            **report,
            **{f"micro_{key}": metrics["micro"][key] for key in ("precision", "recall", "f1")},
        }
    )
    print(f"\n{name}: same thresholds + boundary fix")
    print_metrics(metrics)

fixed_ensemble = vote_predictions(fixed_predictions, MIN_VOTES)
fixed_ensemble_metrics = evaluate_records(fixed_ensemble)
fixed_ensemble_dir = BOUNDARY_DIR / "ensemble"
write_jsonl(fixed_ensemble_dir / "predictions_with_scores.jsonl", fixed_ensemble)
write_jsonl(
    fixed_ensemble_dir / "predictions.jsonl",
    competition_predictions(fixed_ensemble),
)
write_json(fixed_ensemble_dir / "metrics.json", fixed_ensemble_metrics)

boundary_report["ensemble"] = {
    "min_votes": MIN_VOTES,
    "metrics": fixed_ensemble_metrics,
}
write_json(BOUNDARY_DIR / "summary.json", boundary_report)
pd.DataFrame(fixed_summary).to_csv(BOUNDARY_DIR / "models_summary.csv", index=False)

print("\nThresholded ensemble + boundary fix")
print_metrics(fixed_ensemble_metrics)
display(pd.DataFrame(fixed_summary))
print("Boundary-fix outputs saved in", BOUNDARY_DIR)



normalized: same thresholds + boundary fix
scope     precision     recall         f1       tp       fp       fn
--------------------------------------------------------------------
ORG          0.8934     0.8229     0.8567     2188      261      471
NAME         0.9178     0.8862     0.9017     2055      184      264
GEO          0.9352     0.8761     0.9047     2383      165      337
micro        0.9157     0.8607     0.8874     6626      610     1072
macro        0.9155     0.8617     0.8877        -        -        -

big_augmented: same thresholds + boundary fix
scope     precision     recall         f1       tp       fp       fn
--------------------------------------------------------------------
ORG          0.8761     0.8353     0.8552     2221      314      438
NAME         0.9199     0.9013     0.9105     2090      182      229
GEO          0.9191     0.8860     0.9023     2410      212      310
micro        0.9047     0.8731     0.8886     6721      708      977
macro       

,model,changed_entities,dropped_after_overlap,micro_precision,micro_recall,micro_f1
0,normalized,51,1,0.915699,0.860743,0.887371
1,big_augmented,43,1,0.904698,0.873084,0.888610


Boundary-fix outputs saved in /Users/tsagoll/AI Talent Hub/Hack/ner_ensemble_runs/mmbert_pair_dev/word_boundary_fix


## 7. Сравнение стратегий ансамбля на dev

Помимо строгого пересечения сравниваются union и class-wise ансамбль. Class-wise вариант берёт каждый класс у модели, указанной в `LABELWISE_MODEL_BY_CLASS`. Сравнение проводится после тех же thresholds и boundary fix.


In [ ]:
def labelwise_predictions(predictions_by_model, model_by_label, source_records=None):
    if source_records is None:
        source_records = dev_records
    indexed = {
        model_name: {record["hash"]: record["entities"] for record in records}
        for model_name, records in predictions_by_model.items()
    }
    output = []
    for record in source_records:
        record_hash = record["hash"]
        entities = []
        for label in LABELS:
            source_model = model_by_label[label]
            entities.extend(
                dict(entity)
                for entity in indexed[source_model][record_hash]
                if entity["label"] == label
            )
        entities.sort(key=lambda entity: (entity["start"], entity["end"], entity["label"]))
        output.append({"hash": record_hash, "entities": entities})
    return output


def ensemble_by_mode(predictions_by_model, mode, source_records=None):
    if source_records is None:
        source_records = dev_records
    if mode == "intersection":
        return vote_predictions(
            predictions_by_model,
            len(predictions_by_model),
            source_records,
        )
    if mode == "union":
        return vote_predictions(predictions_by_model, 1, source_records)
    if mode == "labelwise":
        return labelwise_predictions(
            predictions_by_model,
            LABELWISE_MODEL_BY_CLASS,
            source_records,
        )
    raise ValueError(f"Неизвестная стратегия ансамбля: {mode}")


dev_strategy_rows = []
dev_strategy_metrics = {}
strategy_dir = ENSEMBLE_DIR / "strategies_after_boundary_fix"

for mode in ("intersection", "union", "labelwise"):
    predictions = ensemble_by_mode(fixed_predictions, mode, dev_records)
    metrics = evaluate_records(predictions)
    dev_strategy_metrics[mode] = metrics
    write_jsonl(strategy_dir / f"{mode}_predictions.jsonl", competition_predictions(predictions))
    write_json(strategy_dir / f"{mode}_metrics.json", metrics)
    dev_strategy_rows.append(
        {
            "mode": mode,
            **{f"micro_{key}": metrics["micro"][key] for key in ("precision", "recall", "f1")},
            "tp": metrics["micro"]["tp"],
            "fp": metrics["micro"]["fp"],
            "fn": metrics["micro"]["fn"],
        }
    )

dev_strategy_table = pd.DataFrame(dev_strategy_rows).sort_values("micro_f1", ascending=False)
display(dev_strategy_table)
dev_strategy_table.to_csv(strategy_dir / "comparison.csv", index=False)
write_json(strategy_dir / "comparison.json", dev_strategy_metrics)


## 8. Public test и контрактный submission

Модели повторно запускаются на `PUBLIC_TEST_PATH`. Если включена нормализация апострофов, применяется тот же length-preserving normalizer, что использовался для normalized train/dev; символьные offsets поэтому не меняются.

Для каждой модели сохраняются raw scores, thresholded и boundary-fixed predictions. Затем создаются три ансамблевых варианта и основной `submission.jsonl`, выбранный через `PUBLIC_SUBMISSION_MODE`. Контрактный файл содержит только `hash` и массив `entities` с полями `label`, `start`, `end`.


In [ ]:
public_input_records = read_records(PUBLIC_TEST_PATH, require_entities=False)
public_records = [dict(record) for record in public_input_records]
normalization_report = {
    "enabled": NORMALIZE_PUBLIC_APOSTROPHES,
    "records": len(public_records),
    "changed_documents": 0,
    "replacement_counts": {},
    "length_preserving": True,
}

if NORMALIZE_PUBLIC_APOSTROPHES:
    normalizer_path = next(INPUT_ROOT.rglob("normalize_apostrophes.py"))
    if str(normalizer_path.parent) not in sys.path:
        sys.path.insert(0, str(normalizer_path.parent))
    from normalize_apostrophes import normalize_text

    replacement_counts = Counter()
    normalized_records = []
    for record in public_input_records:
        normalized_text, changes = normalize_text(record["text"], preserve_english=True)
        if len(normalized_text) != len(record["text"]):
            raise AssertionError("Нормализация public test изменила длину текста")
        normalization_report["changed_documents"] += int(normalized_text != record["text"])
        replacement_counts.update(changes)
        normalized_records.append({"hash": record["hash"], "text": normalized_text})
    public_records = normalized_records
    normalization_report["replacement_counts"] = dict(replacement_counts)

public_records_by_hash = {record["hash"]: record for record in public_records}
write_json(PUBLIC_TEST_DIR / "apostrophe_normalization_report.json", normalization_report)
print("Public records:", len(public_records))
print("Apostrophe normalization:", normalization_report)

public_raw_predictions = {}
public_thresholded_predictions = {}
public_fixed_predictions = {}
public_boundary_reports = {}

for spec in MODEL_SPECS:
    name = spec["name"]
    raw, inference_config = predict_records_with_model(spec, public_records, "public_test")
    thresholded = apply_thresholds(raw, best_thresholds[name])
    fixed, boundary_report = apply_word_boundary_fix(
        thresholded,
        public_records_by_hash,
    )

    public_raw_predictions[name] = raw
    public_thresholded_predictions[name] = thresholded
    public_fixed_predictions[name] = fixed
    public_boundary_reports[name] = boundary_report

    model_output_dir = PUBLIC_TEST_DIR / name
    write_jsonl(model_output_dir / "raw_predictions_with_scores.jsonl", raw)
    write_jsonl(model_output_dir / "thresholded_predictions_with_scores.jsonl", thresholded)
    write_jsonl(model_output_dir / "fixed_predictions_with_scores.jsonl", fixed)
    write_jsonl(model_output_dir / "fixed_predictions.jsonl", competition_predictions(fixed))
    write_json(model_output_dir / "inference_config.json", inference_config)
    write_json(model_output_dir / "boundary_fix_report.json", boundary_report)


def validate_submission(predictions, inputs):
    if len(predictions) != len(inputs):
        raise ValueError("Submission содержит неправильное число строк")
    for prediction, source in zip(predictions, inputs, strict=True):
        if set(prediction) != {"hash", "entities"}:
            raise ValueError("В submission допустимы только hash и entities")
        if prediction["hash"] != source["hash"]:
            raise ValueError("Порядок или hash строк submission не совпадает с public input")
        seen = set()
        for entity in prediction["entities"]:
            if set(entity) != {"label", "start", "end"}:
                raise ValueError("Entity должна содержать только label/start/end")
            label, start, end = entity["label"], entity["start"], entity["end"]
            if label not in LABELS:
                raise ValueError(f"Недопустимый класс: {label}")
            if (
                not isinstance(start, int)
                or isinstance(start, bool)
                or not isinstance(end, int)
                or isinstance(end, bool)
                or not 0 <= start < end <= len(source["text"])
            ):
                raise ValueError(f"Некорректные offsets для {source['hash']}")
            key = (label, start, end)
            if key in seen:
                raise ValueError(f"Дублирующая entity для {source['hash']}")
            seen.add(key)


public_ensembles = {}
public_summary = {
    "records": len(public_records),
    "thresholds": best_thresholds,
    "boundary_fix": public_boundary_reports,
    "ensemble_modes": {},
    "selected_submission_mode": PUBLIC_SUBMISSION_MODE,
    "labelwise_model_by_class": LABELWISE_MODEL_BY_CLASS,
}

for mode in ("intersection", "union", "labelwise"):
    scored_predictions = ensemble_by_mode(public_fixed_predictions, mode, public_records)
    submission_predictions = competition_predictions(scored_predictions)
    validate_submission(submission_predictions, public_input_records)
    public_ensembles[mode] = scored_predictions

    write_jsonl(PUBLIC_TEST_DIR / f"{mode}_predictions_with_scores.jsonl", scored_predictions)
    write_jsonl(PUBLIC_TEST_DIR / f"submission_{mode}.jsonl", submission_predictions)
    label_counts = Counter(
        entity["label"]
        for record in submission_predictions
        for entity in record["entities"]
    )
    public_summary["ensemble_modes"][mode] = {
        "predicted_entities": sum(label_counts.values()),
        "by_label": dict(label_counts),
    }

selected_submission = competition_predictions(public_ensembles[PUBLIC_SUBMISSION_MODE])
validate_submission(selected_submission, public_input_records)
submission_path = PUBLIC_TEST_DIR / "submission.jsonl"
write_jsonl(submission_path, selected_submission)
write_json(PUBLIC_TEST_DIR / "summary.json", public_summary)

display(pd.DataFrame(public_summary["ensemble_modes"]).T)
print("Selected mode:", PUBLIC_SUBMISSION_MODE)
print("Leaderboard submission:", submission_path)


## Структура результатов

```text
/kaggle/working/ner_ensemble_runs/<experiment>/
├── configs/
│   ├── experiment_config.json
│   ├── model_1.json
│   └── model_2.json
├── raw/
│   ├── *_predictions_with_scores.jsonl
│   └── *_metrics_no_threshold.json
├── thresholds/
│   ├── *_threshold_metrics.csv
│   ├── *_threshold_metrics.json
│   ├── *_best_thresholds.json
│   ├── *_thresholded_predictions.jsonl
│   └── *_thresholded_metrics.json
├── ensemble/
│   ├── ensemble_predictions.jsonl
│   ├── ensemble_predictions_with_scores.jsonl
│   └── ensemble_metrics.json
├── word_boundary_fix/
│   ├── model_1/
│   ├── model_2/
│   ├── ensemble/
│   ├── models_summary.csv
│   └── summary.json
└── public_test/
    ├── normalized/
    ├── big_augmented/
    ├── submission_intersection.jsonl
    ├── submission_union.jsonl
    ├── submission_labelwise.jsonl
    ├── submission.jsonl
    └── summary.json
```

Файлы `predictions.jsonl` содержат только конкурсные поля `label/start/end`; варианты `predictions_with_scores.jsonl` сохраняют вероятности и диагностические поля.
